# Alignment debug
Shows detected landmarks and the canonical target overlaid on both domains.
Use this to tune `_CANONICAL_128` in `kaoanime/utils/align.py` if faces look off.

In [ ]:
import sys
sys.path.insert(0, '..')

import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from kaoanime.utils.align import (
    AlignFaceProcessor, _CANONICAL_128, _canonical,
    _src_pts, _LEFT_EYE_IDX, _RIGHT_EYE_IDX, _NOSE_IDX, _L_MOUTH_IDX, _R_MOUTH_IDX,
)
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
from kaoanime.utils.align import _ensure_model

In [ ]:
ROOT_A = Path('/beta/home/madorskii/datasets/CelebA/img_align_celeba/img_align_celeba')
ROOT_B = Path('/beta/home/madorskii/datasets/alignedanimefaces/safebooru_jpeg')

IMAGE_SIZE = 128
N_IMAGES   = 6
SEED       = 0

# Canonical point colors: left-eye, right-eye, nose, left-mouth, right-mouth
COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']  # R, B, G, O, P
LABELS = ['L-eye', 'R-eye', 'Nose', 'L-mouth', 'R-mouth']

In [ ]:
def collect(root, n, seed=0):
    exts = {'.jpg', '.jpeg', '.png'}
    files = [p for p in root.iterdir() if p.suffix.lower() in exts]
    return random.Random(seed).sample(files, min(n, len(files)))

def center_crop(img, size):
    h, w = img.shape[:2]
    s = min(h, w)
    y0, x0 = (h - s) // 2, (w - s) // 2
    return cv2.resize(img[y0:y0+s, x0:x0+s], (size, size), interpolation=cv2.INTER_AREA)

# Build a raw FaceLandmarker for landmark detection (without AlignFaceProcessor wrapper)
options = mp_vision.FaceLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=_ensure_model()),
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1,
)
landmarker = mp_vision.FaceLandmarker.create_from_options(options)
processor  = AlignFaceProcessor()

In [ ]:
def detect_landmarks(rgb: np.ndarray):
    """Return 5 detected src points (x, y) or None."""
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_img)
    if not result.face_landmarks:
        return None
    h, w = rgb.shape[:2]
    return _src_pts(result.face_landmarks[0], h, w)  # shape (5, 2)

def draw_pts(ax, pts, colors, size=8, marker='o'):
    for (x, y), c in zip(pts, colors):
        ax.plot(x, y, marker, color=c, markersize=size, markeredgecolor='white', markeredgewidth=0.8)

## Domain A — detected landmarks vs canonical target
Left column: raw image with detected landmarks (dots) and canonical target positions (crosses).  
Right column: aligned output with canonical target overlaid — check if dots land on crosses.

In [ ]:
files_a = collect(ROOT_A, N_IMAGES, SEED)
canonical_pts = _CANONICAL_128  # 5×2, in 128px space

fig, axes = plt.subplots(N_IMAGES, 3, figsize=(9, N_IMAGES * 3))
fig.suptitle('Domain A: raw | aligned | canonical overlay on aligned', fontsize=12)

for row, path in enumerate(files_a):
    raw = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    src_pts = detect_landmarks(raw)
    aligned = processor.align(raw, IMAGE_SIZE)
    crop    = center_crop(raw, IMAGE_SIZE)
    show    = aligned if aligned is not None else crop

    # col 0: raw with detected landmarks
    ax0 = axes[row, 0]
    ax0.imshow(raw)
    if src_pts is not None:
        draw_pts(ax0, src_pts, COLORS)
    else:
        ax0.set_title('NO FACE', color='red', fontsize=8)
    ax0.axis('off')

    # col 1: aligned image
    ax1 = axes[row, 1]
    ax1.imshow(show)
    ax1.axis('off')

    # col 2: aligned image + canonical target points
    ax2 = axes[row, 2]
    ax2.imshow(show)
    draw_pts(ax2, canonical_pts, COLORS, size=10, marker='x')
    ax2.axis('off')

# Legend
patches = [mpatches.Patch(color=c, label=l) for c, l in zip(COLORS, LABELS)]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9)
plt.tight_layout()
plt.show()

## Domain B — canonical target positions overlaid on anime center-crops
Crosses show where the canonical landmarks are expected to land after alignment.  
If anime faces are consistently different, adjust `_CANONICAL_128` in `align.py`.

In [ ]:
files_b = collect(ROOT_B, N_IMAGES * N_IMAGES, SEED)  # larger sample for grid

cols = N_IMAGES
rows = N_IMAGES
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
fig.suptitle('Domain B (anime): center-crop + canonical target points (×)', fontsize=12)

for i, path in enumerate(files_b[:rows * cols]):
    raw  = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    crop = center_crop(raw, IMAGE_SIZE)
    ax   = axes[i // cols, i % cols]
    ax.imshow(crop)
    draw_pts(ax, canonical_pts, COLORS, size=6, marker='x')
    ax.axis('off')

patches = [mpatches.Patch(color=c, label=l) for c, l in zip(COLORS, LABELS)]
fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=9)
plt.tight_layout()
plt.show()

## How to fix misalignment

If the canonical crosses consistently miss the actual eye/mouth positions in domain B:

1. Visually estimate where the landmarks actually sit in the anime crops (pixel coordinates)
2. Open `kaoanime/utils/align.py` and update `_CANONICAL_128`:
   ```python
   _CANONICAL_128 = np.array([
       [LEFT_EYE_X,   LEFT_EYE_Y  ],  # left eye centre
       [RIGHT_EYE_X,  RIGHT_EYE_Y ],  # right eye centre
       [NOSE_X,       NOSE_Y      ],  # nose tip
       [L_MOUTH_X,    L_MOUTH_Y   ],  # left mouth corner
       [R_MOUTH_X,    R_MOUTH_Y   ],  # right mouth corner
   ], dtype=np.float32)
   ```
3. Re-run this notebook to confirm the new canonical matches both domains.